## EDA
Gli obiettivi dell'analisi esplorativa sono:
1. Calcolo percentuale dropout con vari threshold
2. Verifica delle azioni effettuate dagli studenti durante tutto l'anno
3. Analisi esplorativa delle actions, percentuali, sparsità.
4. Analisi Top Actions
5. Correlazione Actions/Dropout
6. Engagement dello studente
7. Collinearità

In [ ]:
# TODO change paths

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
INPUT_DIR = "/datasets/unitelma"
OUTPUT_DIR = "/notebooks/outputs/eda"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### 1 - Calcolo percentuale dropout per vari threshold

In [4]:
thresholds_normal = np.arange(0.1, 1.0, 0.1) # [0.1, 0.2, ..., 0.9]
thresholds_extreme = np.arange(0.90, 1.00, 0.01) # [0.90, 0.91, ..., 0.99]

In [5]:
results_normal = []
results_extreme = []

# Lista dei corsi e dei nomi file (gestisce anche il file all_timeseries.csv
files_to_process = [(str(i), f"timeseries_{i}.csv") for i in range(1, 14)]
files_to_process.append(("all", "all_timeseries.csv"))

for course, file_name in files_to_process:
    file_path = os.path.join(INPUT_DIR, file_name)
    
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        continue
        
    # Caricamento solo delle colonne necessarie
    df = pd.read_csv(file_path, usecols=['dropout'])
    
    # Calcolo binarizzazione per threshold normal
    for th in thresholds_normal:
        pct = (df['dropout'] >= th).mean() * 100
        results_normal.append({
            'course': course, 
            'threshold': round(th, 1), 
            'pct_dropout': round(pct, 2)
        })
        
    # 2. Calcolo binarizzazione per threshold extreme
    for th in thresholds_extreme:
        pct = (df['dropout'] >= th).mean() * 100
        results_extreme.append({
            'course': course, 
            'threshold': round(th, 2), 
            'pct_dropout': round(pct, 2)
        })

# Creazione DataFrame e salvataggio
pd.DataFrame(results_normal).to_csv(os.path.join(OUTPUT_DIR, "dropout_percentage.csv"), index=False)
pd.DataFrame(results_extreme).to_csv(os.path.join(OUTPUT_DIR, "dropout_percentage_extreme.csv"), index=False)

print(f"[SUCCESS] Reports saved in {OUTPUT_DIR}")

[SUCCESS] Reports saved in /notebooks/outputs/eda


In [15]:
# Generazione di grafici line chart e heatmap dei risultati
INPUT_PLOT = "/notebooks/outputs/eda"
OUTPUT_PLOT = "/notebooks/plots"
os.makedirs(OUTPUT_PLOT, exist_ok=True)

In [16]:
df_normal = pd.read_csv(os.path.join(INPUT_PLOT, "dropout_percentage.csv"))
df_extreme = pd.read_csv(os.path.join(INPUT_PLOT, "dropout_percentage_extreme.csv"))

In [26]:
def plot_line_with_highlight(df, filename_suffix, title_suffix):
    plt.figure(figsize=(12, 7)) # Leggermente più largo per far spazio alla legenda
    
    # Creiamo un plot automatico dove ogni corso ha un colore diverso
    # Usiamo la palette 'tab20' che ha abbastanza colori per 14 elementi
    ax = sns.lineplot(
        data=df[df['course'] != 'all'], # Plottiamo prima i corsi singoli
        x='threshold', 
        y='pct_dropout', 
        hue='course',
        palette='tab20',
        linewidth=1.5,
        alpha=0.8
    )
    
    subset_all = df[df['course'] == 'all']
    plt.plot(
        subset_all['threshold'], 
        subset_all['pct_dropout'], 
        color='red',
        linewidth=4, 
        label='All Courses'
    )
    
    plt.title(f"Decadimento del Dropout per Threshold ({title_suffix})", fontsize=14)
    plt.xlabel("Threshold", fontsize=12)
    plt.ylabel("% Dropout", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Spostiamo la legenda fuori dal grafico sulla destra per non coprire le linee
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Course")
    plt.tight_layout()
    
    plt.savefig(os.path.join(OUTPUT_PLOT, f"pct_dropout_line_{filename_suffix}.png"), dpi=300)
    plt.close()
    
def plot_heatmap(df, filename_suffix, title_suffix):
    # Pivot della tabella per avere i corsi sulle righe e i threshold sulle colonne
    pivot_df = df.pivot(index='course', columns='threshold', values='pct_dropout')
    
    # Assicuriamoci che 'all' sia in cima o in fondo per visibilità (es. mettiamolo alla fine)
    idx_order = [c for c in pivot_df.index if c != 'all'] + ['all']
    pivot_df = pivot_df.reindex(idx_order)
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_df, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': '% Dropout'})
    plt.title(f"Heatmap % Dropout per Corso e Threshold ({title_suffix})", fontsize=14)
    plt.xlabel("Threshold", fontsize=12)
    plt.ylabel("Course", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PLOT, f"pct_dropout_heatmap_{filename_suffix}.png"), dpi=300)
    plt.close()

In [27]:
# Generazione Plot - Soglie Normali
plot_line_with_highlight(df_normal, "normal", "Normal: 0.1 - 0.9")
plot_heatmap(df_normal, "normal", "Normal: 0.1 - 0.9")

# Generazione Plot - Soglie Estreme
plot_line_with_highlight(df_extreme, "extreme", "Extreme: 0.90 - 0.99")
plot_heatmap(df_extreme, "extreme", "Extreme: 0.90 - 0.99")

print(f"[SUCCESS] Plot generati e salvati in {OUTPUT_PLOT}")

/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1075: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to 

[SUCCESS] Plot generati e salvati in /notebooks/plots


### 2 - Verifica delle azioni effettuate dagli studenti durante tutto l'anno
In questa sezione andremo a verificare durante tutto l'anno quanti giorni, per ogni studente, è stata
effettuata l'azione

In [9]:
INPUT_PROCESSED = "/notebooks/data/processed"
OUTPUT_AGGREGATED = "/notebooks/data/processed_aggregated"
os.makedirs(OUTPUT_AGGREGATED, exist_ok=True)

In [4]:
files_to_process = []

for i in range(1, 14):
    in_name = f"processed_timeseries_{i}.csv"
    out_name = f"processed_abs_timeseries_{i}.csv"
    files_to_process.append((in_name, out_name))

files_to_process.append(("processed_all_timeseries.csv", "processed_abs_timeseries_all.csv"))

def aggregate_active_days(input_path, output_path):
    print(f"[LOG] Aggregating: {os.path.basename(input_path)}...")
    df = pd.read_csv(input_path)
    
    metadata_cols = ['student_id', 'course_id', 'day', 'dropout']
    action_cols = [col for col in df.columns if col not in metadata_cols]
    
    # Binarizzazione: > 0 diventa 1
    df[action_cols] = (df[action_cols] > 0).astype(int)
    
    # Setup aggregazione solo per le azioni e il dropout
    agg_funcs = {col: 'sum' for col in action_cols}
    agg_funcs['dropout'] = 'first'
    
    # La chiave di raggruppamento DEVE includere anche il course_id 
    # per distinguere lo stesso studente in corsi diversi.
    if 'course_id' in df.columns:
        group_keys = ['course_id', 'student_id']
    else:
        group_keys = ['student_id'] # Fallback se manca nei file singoli
        
    df_agg = df.groupby(group_keys).agg(agg_funcs).reset_index()
    
    df_agg.to_csv(output_path, index=False)
    print(f"[OK] Saved {output_path} - Shape: {df_agg.shape}")

In [5]:
for in_file, out_file in files_to_process:
    input_path = os.path.join(INPUT_PROCESSED, in_file)
    output_path = os.path.join(OUTPUT_AGGREGATED, out_file)
    
    if os.path.exists(input_path):
        aggregate_active_days(input_path, output_path)
    else:
        print(f"[ERROR] File non trovato: {input_path}")

print("[SUCCESS] Aggregation Completed!")

[LOG] Aggregating: processed_timeseries_1.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_timeseries_1.csv - Shape: (113, 100)
[LOG] Aggregating: processed_timeseries_2.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_timeseries_2.csv - Shape: (250, 100)
[LOG] Aggregating: processed_timeseries_3.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_timeseries_3.csv - Shape: (75, 100)
[LOG] Aggregating: processed_timeseries_4.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_timeseries_4.csv - Shape: (676, 100)
[LOG] Aggregating: processed_timeseries_5.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_timeseries_5.csv - Shape: (269, 100)
[LOG] Aggregating: processed_timeseries_6.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_timeseries_6.csv - Shape: (469, 100)
[LOG] Aggregating: processed_timeseries_7.csv...
[OK] Saved /notebooks/data/processed_aggregated/processed_abs_ti

In [12]:
input_file = os.path.join(OUTPUT_AGGREGATED, "processed_abs_timeseries_all.csv")
output_file = os.path.join(OUTPUT_DIR, "mean_abs_actions.csv")

df_all = pd.read_csv(input_file)

exclude_cols = ['course_id', 'student_id', 'dropout']
action_cols = [col for col in df_all.columns if col not in exclude_cols]

df_actions_only = df_all[action_cols]

print(f"[LOG] Colonne identificate per la media: {len(action_cols)}")
print(f"[LOG] Shape del dataframe filtrato: {df_actions_only.shape}")

series_means = df_actions_only.mean()

# 4. Ordinamento decrescente e selezione delle prime 10
top_10_means = series_means.sort_values(ascending=False).head(10)

# 5. Trasformazione in DataFrame per il salvataggio
# Lo salviamo in formato "Action, Mean" per chiarezza, data la natura della selezione
df_top_10 = top_10_means.reset_index()
df_top_10.columns = ['action', 'mean_value']

# 6. Salvataggio in OUTPUT_DIR
df_top_10.to_csv(output_file, index=False)

print(f"[OK] File creato con le top 10 azioni: {output_file}")
print(top_10_means) # Visualizza a schermo i risultati

[LOG] Colonne identificate per la media: 97
[LOG] Shape del dataframe filtrato: (5156, 97)
[OK] File creato con le top 10 azioni: /notebooks/outputs/eda/mean_abs_actions.csv
view               64.063227
called              0.839604
submitted           0.697828
graded              0.643134
launched            0.517261
view forum          0.467029
view discussion     0.369666
started             0.276959
reviewed            0.240497
created             0.202095
dtype: float64


### 3 - Analisi esplorativa delle azioni

In [ ]:
INPUT_PROCESSED = "notebooks/data/processed_abs"
EDA_OUT_DIR = "/notebooks/outputs/eda"
PLOT_OUT_DIR = "/notebooks/plots"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Lista dei corsi e nome file aggregato
courses = [str(i) for i in range(1, 14)] + ['all']
dead_actions_data = []

print("[LOG] Computing actions at 0...")

for course in courses:
    file_path = os.path.join(INPUT_PROCESSED, f"processed_abs_timeseries_{course}.csv")
    if not os.path.exists(file_path):
        print(f"[WARNING] File not found: {file_path}")
        continue
        
    df = pd.read_csv(file_path)
    
    # Isolare le colonne delle azioni
    metadata_cols = ['student_id', 'course_id', 'dropout']
    action_cols = [col for col in df.columns if col not in metadata_cols]
    
    # Calcolo della somma totale per ogni azione
    # (Se la somma è 0, nessuno studente in nessun giorno ha compiuto quell'azione)
    action_sums = df[action_cols].sum()
    
    # Filtrare le azioni con somma pari a 0
    dead_actions = action_sums[action_sums == 0].index.tolist()
    
    dead_actions_data.append({
        'course': course,
        'num_dead_actions': len(dead_actions),
        'dead_actions_list': ", ".join(dead_actions)
    })

# 1. Salvataggio su CSV
df_dead = pd.DataFrame(dead_actions_data)
csv_path = os.path.join(EDA_OUT_DIR, "dead_actions.csv")
df_dead.to_csv(csv_path, index=False)
print(f"[OK] Salvato report Azioni Morte in {csv_path}")

[LOG] Computing actions at 0...
[OK] Salvato report Azioni Morte in /notebooks/outputs/eda/dead_actions.csv


In [4]:
# 2. Creazione del Bar Plot
plt.figure(figsize=(10, 6))
# Evidenziamo il dataset "all" con un colore diverso rispetto ai corsi singoli
colors = ['grey' if c != 'all' else 'red' for c in df_dead['course']]

sns.barplot(data=df_dead, x='course', y='num_dead_actions', palette=colors)
plt.title("Numero di Azioni Mai Effettuate (Dead Actions) per Corso", fontsize=14)
plt.xlabel("Corso", fontsize=12)
plt.ylabel("Numero di Azioni Morte", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plot_path = os.path.join(PLOT_OUT_DIR, "dead_actions_count.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Salvato plot in {plot_path}")

[OK] Salvato plot in /notebooks/plots/dead_actions_count.png


### 4 - Top Actions

In [5]:
courses = [str(i) for i in range(1, 14)] + ['all']
top_actions_data = []
all_courses_pct = {}

print("[LOG] Computing Top Actions...")

for course in courses:
    file_path = os.path.join(INPUT_PROCESSED, f"processed_abs_timeseries_{course}.csv")
    if not os.path.exists(file_path):
        continue
        
    df = pd.read_csv(file_path)
    metadata_cols = ['student_id', 'course_id', 'dropout']
    action_cols = [col for col in df.columns if col not in metadata_cols]
    
    n_students = len(df)
    
    # In processed_abs le azioni indicano i "giorni attivi". 
    # Per calcolare % di studenti che l'hanno fatta ALMENO una volta,
    # binarizziamo temporaneamente le somme per studente (>0 diventa 1)
    students_did_action = (df[action_cols] > 0).sum()
    pct_students = (students_did_action / n_students) * 100
    
    # Salviamo le percentuali per l'heatmap
    all_courses_pct[course] = pct_students
    
    # Top 5 per il report testuale
    top_5 = pct_students.nlargest(5)
    
    top_actions_data.append({
        'course': course,
        'top_1': f"{top_5.index[0]} ({top_5.iloc[0]:.1f}%)",
        'top_2': f"{top_5.index[1]} ({top_5.iloc[1]:.1f}%)",
        'top_3': f"{top_5.index[2]} ({top_5.iloc[2]:.1f}%)",
        'top_4': f"{top_5.index[3]} ({top_5.iloc[3]:.1f}%)",
        'top_5': f"{top_5.index[4]} ({top_5.iloc[4]:.1f}%)"
    })

# 1. Salvataggio CSV
df_top = pd.DataFrame(top_actions_data)
csv_path = os.path.join(EDA_OUT_DIR, "top_actions.csv")
df_top.to_csv(csv_path, index=False)
print(f"[OK] Salvato report Top Actions in {csv_path}")

[LOG] Computing Top Actions...
[OK] Salvato report Top Actions in /notebooks/outputs/eda/top_actions.csv


In [6]:
df_all_pct = pd.DataFrame(all_courses_pct) # Colonne = corsi, Indice = azioni
top_10_global_actions = df_all_pct['all'].nlargest(10).index

# Filtriamo solo le top 10 e riordiniamo le colonne per avere 'all' alla fine
df_heatmap = df_all_pct.loc[top_10_global_actions, [str(i) for i in range(1, 14)] + ['all']]

plt.figure(figsize=(14, 8))
sns.heatmap(df_heatmap, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': '% Studenti'})
plt.title("Adozione delle Top 10 Azioni Globali per Corso (% Studenti)", fontsize=14)
plt.ylabel("Azione", fontsize=12)
plt.xlabel("Corso", fontsize=12)
plt.tight_layout()

plot_path = os.path.join(PLOT_OUT_DIR, "top_actions_heatmap.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Salvato plot in {plot_path}")

[OK] Salvato plot in /notebooks/plots/top_actions_heatmap.png


### 5 - Correlazione Azione/Dropout

In [7]:
print("[LOG] Calcolo correlazione Azione-Dropout...")

df = pd.read_csv("/notebooks/data/processed_aggregated/processed_abs_timeseries_all.csv")

# Isolare le colonne delle azioni (escludiamo id)
metadata_cols = ['student_id', 'course_id'] # Teniamo 'dropout' per la correlazione
action_cols = [col for col in df.columns if col not in metadata_cols and col != 'dropout']

# 1. Calcolo della correlazione
# df[action_cols + ['dropout']].corr() calcola la matrice, 
# selezioniamo solo la colonna 'dropout' e rimuoviamo l'autocorrelazione (dropout-dropout)
correlations = df[action_cols + ['dropout']].corr()['dropout'].drop('dropout')

# Ordinare i valori e rimuovere i NaN (es. per le 25 azioni morte la varianza è 0, quindi corr è NaN)
correlations = correlations.dropna().sort_values()

# Salvataggio CSV
corr_df = correlations.reset_index()
corr_df.columns = ['action', 'correlation_with_dropout']
csv_path = os.path.join(EDA_OUT_DIR, "action_dropout_correlation.csv")
corr_df.to_csv(csv_path, index=False)
print(f"[OK] Salvato report Correlazioni in {csv_path}")

# 2. Creazione Plot (Top 15 Negative vs Top 15 Positive/Least Negative)
# Le correlazioni negative indicano che fare l'azione riduce il dropout (comportamento "virtuoso")
# Prendo i due estremi della lista ordinata
top_15_negative = correlations.head(15)
top_15_positive = correlations.tail(15)

# Concateniamo per il plot
plot_data = pd.concat([top_15_negative, top_15_positive])

[LOG] Calcolo correlazione Azione-Dropout...
[OK] Salvato report Correlazioni in /notebooks/outputs/eda/action_dropout_correlation.csv


In [8]:
plt.figure(figsize=(12, 10))
# Colori: verde per le negative (buone), rosso per le positive (cattive o meno impattanti)
colors = ['forestgreen' if val < 0 else 'indianred' for val in plot_data.values]

sns.barplot(x=plot_data.values, y=plot_data.index, palette=colors)
plt.title("Correlazione Point-Biserial tra Giorni Attivi per Azione e Dropout", fontsize=14)
plt.xlabel("Coefficiente di Correlazione (Negativo = Riduce Dropout)", fontsize=12)
plt.ylabel("Azione", fontsize=12)
plt.axvline(0, color='black', linewidth=1, linestyle='--')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()

plot_path = os.path.join(PLOT_OUT_DIR, "correlation_dropout.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Salvato plot in {plot_path}")

[OK] Salvato plot in /notebooks/plots/correlation_dropout.png


### 6 - Engagement dello studente

In [14]:
print("[LOG] Correzione della colonna dropout e ricalcolo Engagement...")

df = pd.read_csv("/notebooks/data/processed_aggregated/processed_abs_timeseries_all.csv")

df['dropout'] = (df['dropout'] > 0.5).astype(int)
print(f"[LOG] Target binarizzato. Nuova distribuzione:\n{df['dropout'].value_counts()}")

metadata_cols = ['student_id', 'course_id', 'dropout']
action_cols = [col for col in df.columns if col not in metadata_cols]
df['total_engagement'] = df[action_cols].sum(axis=1)

engagement_stats = df.groupby('dropout')['total_engagement'].describe()
engagement_stats.to_csv(os.path.join(EDA_OUT_DIR, "engagement_stats_fixed.csv"))
print("[OK] Statistiche rigenerate correttamente (0 e 1).")

[LOG] Correzione della colonna dropout e ricalcolo Engagement...
[LOG] Target binarizzato. Nuova distribuzione:
dropout
1    4103
0    1053
Name: count, dtype: int64
[OK] Statistiche rigenerate correttamente (0 e 1).


In [15]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='dropout', y='total_engagement', palette=['forestgreen', 'indianred'])

plt.title("Distribuzione dell'Engagement Totale (Somma Azioni) per Classe", fontsize=14)
plt.xlabel("Classe (0 = Completato, 1 = Abbandonato)", fontsize=12)
plt.ylabel("Total Engagement (Giorni-Azione)", fontsize=12)

# Tagliamo gli outlier più estremi per pulizia visiva, ma in modo dinamico
y_max = df['total_engagement'].quantile(0.98)
plt.ylim(0, y_max)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plot_path = os.path.join(PLOT_OUT_DIR, "engagement_distribution_fixed.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Plot salvato in {plot_path}")

[OK] Plot salvato in /notebooks/plots/engagement_distribution_fixed.png


### 7 - Collinearità

In [6]:
print("[LOG] Calcolo della collinearità e salvataggio coppie...")

df = pd.read_csv("/notebooks/data/processed_aggregated/processed_abs_timeseries_all.csv")
metadata_cols = ['student_id', 'course_id', 'dropout']
action_cols = [col for col in df.columns if col not in metadata_cols]

# Rimuoviamo le azioni "morte" (somma = 0)
action_sums = df[action_cols].sum()
active_actions = action_sums[action_sums > 0].index.tolist()

# Calcoliamo la matrice di correlazione
corr_matrix = df[active_actions].corr()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Creazione del CSV con le coppie altamente correlate
pairs = []
# Iteriamo sulle colonne e troviamo le feature correlate > 0.70 o < -0.70
for col in upper_tri.columns:
    for row in upper_tri.index:
        val = upper_tri.loc[row, col]
        if pd.notna(val) and abs(val) > 0.70:
            pairs.append({
                'feature_1': row,
                'feature_2': col,
                'correlation': round(val, 4)
            })

if pairs:
    df_pairs = pd.DataFrame(pairs).sort_values(by='correlation', ascending=False)
    csv_path = os.path.join(EDA_OUT_DIR, "high_collinearity_pairs.csv")
    df_pairs.to_csv(csv_path, index=False)
    print(f"[OK] Trovate {len(pairs)} coppie altamente correlate. Salvato report in {csv_path}")
    
    # Uniamo feature_1 e feature_2 per avere la lista univoca da plottare
    cols_to_plot = list(set(df_pairs['feature_1']).union(set(df_pairs['feature_2'])))
    plt.figure(figsize=(14, 12))
    sns.heatmap(df[cols_to_plot].corr(), annot=True, fmt=".2f", cmap="coolwarm", 
                cbar_kws={'label': 'Pearson Correlation'}, square=True)
    plt.title("Mappa di Collinearità (Solo feature con |corr| > 0.70)", fontsize=14)
    plt.tight_layout()
    
    plot_path = os.path.join(PLOT_OUT_DIR, "collinearity_heatmap.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f"[OK] Plot salvato in {plot_path}")
else:
    print("[LOG] Nessuna feature ha una correlazione |corr| > 0.70. CSV non generato.")

[LOG] Calcolo della collinearità e salvataggio coppie...
[OK] Trovate 72 coppie altamente correlate. Salvato report in /notebooks/outputs/eda/high_collinearity_pairs.csv
[OK] Plot salvato in /notebooks/plots/collinearity_heatmap.png
